# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faheem-danish/internship-starter-1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from google.colab import userdata
token = userdata.get("HF_TOKEN")
print("Token loaded:", token is not None)

Token loaded: True


In [4]:
march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=token
)

print("March file:", march_file)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

March file: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [5]:
march_df = pd.read_parquet(march_file)

In [6]:
# ------------------------------------------------------------------
# Aggregate page-day → one row per client + content (March window)
# ------------------------------------------------------------------
model_df = march_df.copy()

# GSC features (label-derived suspects)
model_df["impressions"]   = pd.to_numeric(model_df["gsc_impressions"],   errors="coerce").fillna(0)
model_df["clicks"]        = pd.to_numeric(model_df["gsc_clicks"],        errors="coerce").fillna(0)
model_df["avg_position"]  = pd.to_numeric(model_df["gsc_avg_position"],  errors="coerce")
model_df["ctr_pct"]       = np.where(model_df["impressions"] > 0,
                                     model_df["clicks"] / model_df["impressions"] * 100, 0)

In [7]:
# GA4 + engagement features (independent of the GSC label rule)
model_df["ga4_sessions"]          = pd.to_numeric(model_df["ga4_sessions"],          errors="coerce")
model_df["ga4_pageviews"]         = pd.to_numeric(model_df["ga4_pageviews"],         errors="coerce")
model_df["ga4_users"]             = pd.to_numeric(model_df["ga4_users"],             errors="coerce")
model_df["ga4_engaged_sessions"]  = pd.to_numeric(model_df["ga4_engaged_sessions"],  errors="coerce")
model_df["scroll_events"]         = pd.to_numeric(model_df["scroll_events"],         errors="coerce")

model_df = model_df.groupby(["client_hash_id", "content_hash_id"], as_index=False).agg({
    "impressions": "sum",
    "clicks": "sum",
    "avg_position": "mean",
    "ga4_sessions": "sum",
    "ga4_pageviews": "sum",
    "ga4_users": "sum",
    "ga4_engaged_sessions": "sum",
    "scroll_events": "sum"
})

model_df["ctr_pct"] = np.where(model_df["impressions"] > 0,
                               model_df["clicks"] / model_df["impressions"] * 100, 0)

In [8]:
# ------------------------------------------------------------------
# Target = same W05 baseline rule
# ------------------------------------------------------------------
model_df["eligible"] = (
    (model_df["impressions"] >= 500) &
    (model_df["avg_position"] >= 11)
)

model_df["baseline_score"] = (
    model_df["impressions"] *
    np.where(model_df["avg_position"] >= 21, 1.5, 1.0)
)

y = model_df["eligible"].astype(int)

print("Modeling rows:", len(model_df))
print("Eligible rate:", round(y.mean(), 4))

Modeling rows: 331437
Eligible rate: 0.0638


In [9]:
# ------------------------------------------------------------------
# Client-grouped split (same honest design as W05)
# ------------------------------------------------------------------
groups = model_df["client_hash_id"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(model_df, y, groups=groups))

print("Train clients:", model_df.iloc[train_idx]["client_hash_id"].nunique())
print("Test clients: ", model_df.iloc[test_idx]["client_hash_id"].nunique())
print("Overlap:      ", len(
    set(model_df.iloc[train_idx]["client_hash_id"]) &
    set(model_df.iloc[test_idx]["client_hash_id"])
))

Train clients: 44
Test clients:  11
Overlap:       0


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*
Finding 1 — The Refresh Effect (Finding #4, page 9)
The paper reports that content in the 365+ day age bucket which was "refreshed within 30 days" shows a 3.2× health boost (from 10.7 to 34.5) and 57× more impressions (from 71 to 4,039) than stale content of the same age. This is presented as one of the strongest measured levers in the portfolio.
My methodology question: How is "refreshed" defined, and does the comparison control for pre-refresh visibility? If "refreshed" is inferred from days_since_update ≤ 30, that field is independent of the health-score formula, so it is not strict label leakage. However, the 57× impression boost compares a small group of refreshed old pages against a much larger stale group. The paper itself notes the 361+ bucket is unstable (283 growing vs 1 declining). My question is whether the refreshed pages were matched on their pre-refresh impression level. If the team systematically selected already-visible pages for refresh, the 57× boost measures selection bias (they refreshed winners) rather than the causal effect of updating content. A stronger design would be a pre-post matched comparison or a randomized refresh holdout. Without that, the safe claim is directional — "refreshed pages in this portfolio correlate with higher health" — not causal.
Finding 2 — ML Feature Importance (ML Appendix, page 27)
The paper reports that Average Position is the #1 predictor of Health Score at 43% importance, followed by Impressions at 32%, in a Random Forest trained on the active-content sample.
My methodology question: The target (Health Score) is a composite of exactly those four inputs: Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts). Using the components as features to predict the composite is a textbook label-leakage case — the model is learning the arithmetic of the score, not an independent relationship. The paper commendably discloses this ("health score is partly constructed from inputs such as position and impressions"), but my question is: was a train-without test performed? If Average Position and Impressions are removed from the feature set, does the model's accuracy collapse? If so, the 43%/32% importance is a measure of score arithmetic, not discovered insight. The honest next step would be to train a second model on features that are not in the health-score formula (e.g., content age, word count, days visible, AI sessions) and report both accuracy numbers. That gap is itself a finding about how much of the "prediction" is circular.


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*
W05 already used a client-grouped split (zero client overlap). The dishonesty was not in the split — it was in the features. The target eligible is built directly from impressions ≥ 500 and avg_position ≥ 11. Keeping those columns (and their siblings clicks, ctr_pct) in the feature set lets the model recover the rule exactly.
Before = W05 as executed: grouped split + label-derived GSC features.
After = same grouped split, but features are limited to GA4 and engagement signals that were not used to construct the label.
This is the train-without test from the leakage skill: if the score collapses when suspects are removed, the confession is complete.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ------------------------------------------------------------------
# BEFORE: W05 feature set (contains label-derived suspects)
# ------------------------------------------------------------------
leaky_features = ["impressions", "clicks", "avg_position", "ctr_pct"]
X_leaky = model_df[leaky_features]

# ------------------------------------------------------------------
# AFTER: Only non-label-derived features (GA4 + scroll)
# ------------------------------------------------------------------
honest_features = ["ga4_sessions", "ga4_pageviews", "ga4_users",
                   "ga4_engaged_sessions", "scroll_events"]
X_honest = model_df[honest_features]

# ------------------------------------------------------------------
# Helper: train + evaluate both models on a given feature set
# ------------------------------------------------------------------
def eval_scenario(X, scenario_name):
    X_tr = X.iloc[train_idx]
    X_te = X.iloc[test_idx]
    y_tr = y.iloc[train_idx]
    y_te = y.iloc[test_idx]

    # Logistic Regression (pipeline with imputer — W05 NaN fix)
    lr = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ])
    lr.fit(X_tr, y_tr)
    lr_prob = lr.predict_proba(X_te)[:, 1]

    # Random Forest (pipeline with imputer — fixed from W05 draft)
    rf = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=150, max_depth=8, min_samples_leaf=20,
            random_state=42, n_jobs=-1
        ))
    ])
    rf.fit(X_tr, y_tr)
    rf_prob = rf.predict_proba(X_te)[:, 1]

    baseline_scores = model_df.iloc[test_idx]["baseline_score"].values
    labels = y_te.values

    def p_at_k(scores, labs, k):
        order = np.argsort(-np.asarray(scores))
        return np.asarray(labs)[order[:k]].mean()

    rows = []
    for name, probas in [("Week-4 baseline", baseline_scores),
                         ("Logistic Regression", lr_prob),
                         ("Random Forest", rf_prob)]:
        rows.append({
            "scenario": scenario_name,
            "method": name,
            "precision@20":  round(p_at_k(probas, labels, 20), 4),
            "precision@50":  round(p_at_k(probas, labels, 50), 4),
            "precision@100": round(p_at_k(probas, labels, 100), 4)
        })
    return rows

# ------------------------------------------------------------------
# Run both scenarios
# ------------------------------------------------------------------
before_after = pd.DataFrame(
    eval_scenario(X_leaky, "BEFORE (with leaky GSC features)") +
    eval_scenario(X_honest, "AFTER (without leaky GSC features)")
)

print("Test-set base rate:", round(y.iloc[test_idx].mean(), 4))
display(before_after)

Test-set base rate: 0.0875


,scenario,method,precision@20,precision@50,precision@100
0,BEFORE (with leaky GSC features),Week-4 baseline,0.05,0.16,0.16
1,BEFORE (with leaky GSC features),Logistic Regression,0.05,0.18,0.31
2,BEFORE (with leaky GSC features),Random Forest,1.00,1.00,1.00
3,AFTER (without leaky GSC features),Week-4 baseline,0.05,0.16,0.16
4,AFTER (without leaky GSC features),Logistic Regression,0.00,0.02,0.03
5,AFTER (without leaky GSC features),Random Forest,0.05,0.04,0.08


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*
I run the attack checklist from the hunting-leakage-and-validating skill on my own W05 model.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== LEAKAGE AUDIT CHECKLIST ===\n")

# 1. Label-derived features
print("[1] Label-derived features")
print("    Target: eligible = (impressions >= 500) & (avg_position >= 11)")
print("    Direct components in feature set: impressions, avg_position")
print("    Sibling/derived features: clicks, ctr_pct (computed from impressions)")
print("    Train-WITH  result: RF precision@20 = 1.0000 (suspiciously perfect)")
print("    Train-WITHOUT result: RF precision@20 = see AFTER table above")
print("    Confession: score collapses when suspects removed → label leakage confirmed.\n")

# 2. Future / overlapping windows
print("[2] Future / overlapping windows")
print("    Feature window: March 2026 (aggregated)")
print("    Label window:   March 2026 (same aggregation)")
print("    Status: No future data leakage — BUT the label is a rule on the same window,")
print("            not an independently observed future outcome.\n")

# 3. Decision-derived features (product flags)
print("[3] Decision-derived features")
suspect_cols = ["eligible", "baseline_score", "baseline_action", "reason_code", "rank"]
used = [c for c in suspect_cols if c in model_df.columns]
print("    Product-flag columns present in data:", used if used else "none")
print("    Any used as model inputs: NO")
print("    Status: Clean.\n")

# 4. Base rate
print("[4] Base rate sanity check")
print("    Test-set positive rate:", round(y.iloc[test_idx].mean(), 4))
print("    Any reported accuracy must be judged against this base rate.\n")

# 5. Top-feature sanity check
print("[5] Top-feature sanity check")
rf_leaky = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=150, max_depth=8, min_samples_leaf=20,
        random_state=42, n_jobs=-1
    ))
])
rf_leaky.fit(X_leaky.iloc[train_idx], y.iloc[train_idx])
imp_leaky = pd.Series(
    rf_leaky.named_steps["model"].feature_importances_,
    index=leaky_features
).sort_values(ascending=False)
print("    Importances WITH suspects:")
print(imp_leaky)
print("    -> avg_position + impressions dominate, matching the label construction.")
print("    -> 'Too good' investigated, not celebrated.\n")

# 6. Out-of-fold / grouped split
print("[6] Split design")
print("    Split type: GroupShuffleSplit by client_hash_id")
print("    Client overlap:", len(
    set(model_df.iloc[train_idx]["client_hash_id"]) &
    set(model_df.iloc[test_idx]["client_hash_id"])
))
print("    Status: Honest grouped split confirmed.")

=== LEAKAGE AUDIT CHECKLIST ===

[1] Label-derived features
    Target: eligible = (impressions >= 500) & (avg_position >= 11)
    Direct components in feature set: impressions, avg_position
    Sibling/derived features: clicks, ctr_pct (computed from impressions)
    Train-WITH  result: RF precision@20 = 1.0000 (suspiciously perfect)
    Train-WITHOUT result: RF precision@20 = see AFTER table above
    Confession: score collapses when suspects removed → label leakage confirmed.

[2] Future / overlapping windows
    Feature window: March 2026 (aggregated)
    Label window:   March 2026 (same aggregation)
    Status: No future data leakage — BUT the label is a rule on the same window,
            not an independently observed future outcome.

[3] Decision-derived features
    Product-flag columns present in data: ['eligible', 'baseline_score']
    Any used as model inputs: NO
    Status: Clean.

[4] Base rate sanity check
    Test-set positive rate: 0.0875
    Any reported accuracy must

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*
"When trained on features that include the same signals used to construct the Week-4 baseline rule (impressions and avg_position), the Random Forest recovers that rule exactly (precision@20 = 1.0 on the held-out test clients). This measures consistency between the learned model and the transparent baseline, not independent predictive skill. When the label-derived features are removed, the Random Forest's precision@20 drops to the level shown in the AFTER table, revealing that the apparent strength was driven by feature-label overlap rather than generalizable signal. The observed result is therefore directional and decision-support: it confirms the baseline rule is automatable from the same GSC data, but it does not demonstrate that the model predicts refresh success on unseen signal patterns."


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.